# 进阶实践项目参考答案 02：胸部 X 射线生成：潜空间、覆盖度与记忆风险

把生成结果拆成重建质量、样本多样性和训练样本记忆三个问题。

Kaggle 中先复制到自己的账户，再按任务顺序完成。题目只保留关键填写位置，数据读取、绘图和保存框架已经给出。

## 任务
1. 建立训练与留出特征
2. 比较真实与生成分布
3. 计算样本多样性
4. 完成最近邻审计
5. 解释统计相似与临床真实性的区别

In [1]:
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
SEED=42; OUT=Path('advanced02_results'); OUT.mkdir(exist_ok=True); rng=np.random.default_rng(SEED)
train=rng.normal(size=(500,32)); holdout=rng.normal(.08,1.05,size=(160,32)); generated=rng.normal(.05,1.02,size=(160,32)); generated[:8]=train[:8]+rng.normal(0,.01,(8,32))
nn=NearestNeighbors(n_neighbors=1).fit(train); gen_dist,idx=nn.kneighbors(generated); real_dist,_=nn.kneighbors(holdout)
pair=np.linalg.norm(generated[:80,None]-generated[None,:80],axis=2); diversity=float(pair[np.triu_indices(80,1)].mean()); copied=int((gen_dist[:,0]<np.percentile(real_dist[:,0],1)).sum())
pca=PCA(2,random_state=SEED).fit(np.vstack([holdout,generated])); r=pca.transform(holdout); g=pca.transform(generated)
fig,ax=plt.subplots(1,2,figsize=(10,4)); ax[0].scatter(r[:,0],r[:,1],s=12,alpha=.5,label='real holdout'); ax[0].scatter(g[:,0],g[:,1],s=12,alpha=.5,label='generated'); ax[0].legend(); ax[0].set_title('Feature coverage'); ax[1].hist(real_dist[:,0],20,alpha=.6,label='real-to-train'); ax[1].hist(gen_dist[:,0],20,alpha=.6,label='generated-to-train'); ax[1].legend(); ax[1].set_title('Nearest-neighbor distance'); fig.tight_layout(); fig.savefig(OUT/'advanced02_summary.png',dpi=150); plt.close(fig)
result={'generated_diversity':diversity,'suspiciously_close_samples':copied,'mean_generated_to_train_distance':float(gen_dist.mean()),'mean_holdout_to_train_distance':float(real_dist.mean())}; (OUT/'advanced02_result.json').write_text(json.dumps(result,indent=2),encoding='utf-8'); print(result)


{'generated_diversity': 7.829116665593272, 'suspiciously_close_samples': 8, 'mean_generated_to_train_distance': 5.177721527059062, 'mean_holdout_to_train_distance': 5.6447286853363305}
